In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv(r"C:\Users\bisha\Downloads\All Codes (3)\Classification\WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
df.TotalCharges=pd.to_numeric(df.TotalCharges,errors='coerce')
df.dropna(how='any',inplace=True)
y=df.Churn.values
x=df.drop(['customerID','Churn'],axis=1)
x=pd.get_dummies(x,columns=['gender','Partner','Dependents','PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],drop_first=True)
x['gender_Male']=x['gender_Male'].astype(int)
cols=['Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_Fiber optic', 'InternetService_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No internet service', 'OnlineBackup_Yes',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No internet service', 'StreamingMovies_Yes',
       'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']
for i in cols:
    x[i]=x[i].replace({True:'1',False:'0'}).astype(int)

In [5]:
x.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 30 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   SeniorCitizen                          7032 non-null   int64  
 1   tenure                                 7032 non-null   int64  
 2   MonthlyCharges                         7032 non-null   float64
 3   TotalCharges                           7032 non-null   float64
 4   gender_Male                            7032 non-null   int32  
 5   Partner_Yes                            7032 non-null   int32  
 6   Dependents_Yes                         7032 non-null   int32  
 7   PhoneService_Yes                       7032 non-null   int32  
 8   MultipleLines_No phone service         7032 non-null   int32  
 9   MultipleLines_Yes                      7032 non-null   int32  
 10  InternetService_Fiber optic            7032 non-null   int32  
 11  InternetS

In [6]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=3)
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
x_train=sc.fit_transform(x_train)
x_test=sc.transform(x_test)

In [7]:
from sklearn.tree import DecisionTreeClassifier
model_dt=DecisionTreeClassifier()
model_dt.fit(x_train,y_train)
y_pred_dt=model_dt.predict(x_test)

In [8]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred_dt))

              precision    recall  f1-score   support

          No       0.83      0.79      0.81      1051
         Yes       0.47      0.53      0.50       356

    accuracy                           0.73      1407
   macro avg       0.65      0.66      0.66      1407
weighted avg       0.74      0.73      0.73      1407



In [9]:
from imblearn.over_sampling import RandomOverSampler

In [10]:
oversampler=RandomOverSampler()
x_train_upsampled,y_train_upsampled=oversampler.fit_resample(x_train,y_train)

In [11]:
model_dt.fit(x_train_upsampled,y_train_upsampled)
y_pred_dt2=model_dt.predict(x_test)

In [12]:
print(classification_report(y_test,y_pred_dt2))

              precision    recall  f1-score   support

          No       0.83      0.83      0.83      1051
         Yes       0.50      0.51      0.50       356

    accuracy                           0.75      1407
   macro avg       0.67      0.67      0.67      1407
weighted avg       0.75      0.75      0.75      1407



In [13]:
from imblearn.under_sampling import RandomUnderSampler
undersampler=RandomUnderSampler()

In [14]:
x_train_downsampled,y_train_downsampled=undersampler.fit_resample(x_train,y_train)
model_dt.fit(x_train_downsampled,y_train_downsampled)
y_pred_dt3=model_dt.predict(x_test)
print(classification_report(y_test,y_pred_dt3))

              precision    recall  f1-score   support

          No       0.86      0.69      0.76      1051
         Yes       0.42      0.66      0.51       356

    accuracy                           0.68      1407
   macro avg       0.64      0.68      0.64      1407
weighted avg       0.75      0.68      0.70      1407



In [15]:
from imblearn.combine import SMOTEENN
soteen=SMOTEENN()

In [16]:
x_train_soteen,y_train_soteen=soteen.fit_resample(x_train,y_train)
model_dt.fit(x_train_soteen,y_train_soteen)
y_pred_dt4=model_dt.predict(x_test)
print(classification_report(y_test,y_pred_dt4))

              precision    recall  f1-score   support

          No       0.90      0.71      0.79      1051
         Yes       0.47      0.76      0.58       356

    accuracy                           0.72      1407
   macro avg       0.68      0.74      0.69      1407
weighted avg       0.79      0.72      0.74      1407

